In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install -q \
    pymupdf \
    tiktoken \
    openai \
    faiss-cpu \
    rank-bm25 \
    sentence-transformers \
    fastapi \
    uvicorn \
    pydantic \
    python-dotenv \
    pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 97.2 MB/s eta 0:00:00


In [6]:
!pip uninstall -y openai
!pip install -q groq sentence-transformers faiss-cpu rank-bm25

Found existing installation: openai 2.54.0
Uninstalling openai-2.54.0:
  Successfully uninstalled openai-2.54.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.6 MB/s eta 0:00:00


In [7]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/content/Python_AI_engine")
DATA_DIR = PROJECT_ROOT / "data"
PAPERS_DIR = DATA_DIR / "papers"

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
PAPERS_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Papers:", PAPERS_DIR)

Project: /content/Python_AI_engine
Papers: /content/Python_AI_engine/data/papers


In [8]:
import os
from google.colab import userdata

GROQ_API_KEY = userdata.get("OPENAI_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Groq API key is missing from Colab Secrets.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Groq API key loaded.")

Groq API key loaded.


In [9]:
from groq import Groq

groq_client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

print("Groq client initialized.")

Groq client initialized.


In [10]:
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Explain RAG in one sentence."
        }
    ],
    temperature=0
)

answer = response.choices[0].message.content

print(answer)

Retrieval‑Augmented Generation (RAG) is a technique that combines a language model with a searchable external knowledge base, letting the model first retrieve relevant documents and then generate responses grounded in that retrieved information.


In [11]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


In [12]:
import os

from google.colab import userdata
from groq import Groq
from sentence_transformers import SentenceTransformer


# ============================================================
# 1. Configuration
# ============================================================

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
GROQ_MODEL = "openai/gpt-oss-120b"


# ============================================================
# 2. Load Groq API key from Colab Secrets
# ============================================================

GROQ_API_KEY = userdata.get("OPENAI_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "Groq API key not found. "
        "Add your Groq API key to Colab Secrets."
    )

os.environ["GROQ_API_KEY"] = GROQ_API_KEY


# ============================================================
# 3. Initialize Groq client
# ============================================================

groq_client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)


# ============================================================
# 4. Load local embedding model
# ============================================================

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)


print("AI configuration loaded successfully.")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Embedding size  : {embedding_model.get_sentence_embedding_dimension()}")
print(f"Groq model      : {GROQ_MODEL}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

AI configuration loaded successfully.
Embedding model : BAAI/bge-small-en-v1.5
Embedding size  : 384
Groq model      : openai/gpt-oss-120b


/tmp/ipykernel_1041/3373484462.py:51: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding size  : {embedding_model.get_sentence_embedding_dimension()}")


In [13]:
def embed_text(text: str):
    """
    Convert text into a normalized dense embedding vector.
    """

    return embedding_model.encode(
        text,
        normalize_embeddings=True
    )


text = "Research papers contain structured and unstructured knowledge."

vector = embed_text(text)

print("Embedding generated successfully.")
print("Vector shape:", vector.shape)

Embedding generated successfully.
Vector shape: (384,)


In [14]:
def embed_texts(texts: list[str]):
    """
    Generate normalized embeddings for multiple text chunks.
    """

    return embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )

texts = [
    "Graph neural networks operate on graph-structured data.",
    "Transformers are widely used for natural language processing.",
    "Research papers describe experiments and methodologies."
]

vectors = embed_texts(texts)

print("Number of embeddings:", len(vectors))
print("Embedding dimensions:", vectors.shape[1])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of embeddings: 3
Embedding dimensions: 384


In [15]:
def generate_answer(
    prompt: str,
    temperature: float = 0.0
) -> str:
    """
    Generate a response using the configured Groq LLM.
    """

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature
    )

    return response.choices[0].message.content

answer = generate_answer(
    "Explain Retrieval-Augmented Generation in two sentences."
)

print(answer)

Retrieval‑augmented generation (RAG) combines a large language model with an external knowledge base: the model first queries the database (or search engine) to retrieve relevant documents, then conditions its generation on both the original prompt and the retrieved text. This lets the system produce up‑to‑date, factual, and domain‑specific answers while keeping the flexibility and fluency of a generative model.


In [16]:
from dataclasses import dataclass, field


@dataclass
class DocumentPage:
    page_number: int
    text: str


@dataclass
class ResearchDocument:
    source: str
    pages: list[DocumentPage] = field(default_factory=list)

    @property
    def full_text(self) -> str:
        return "\n\n".join(
            page.text for page in self.pages
        )

    @property
    def page_count(self) -> int:
        return len(self.pages)

In [17]:
import fitz


def extract_pdf(pdf_path: str) -> ResearchDocument:
    """
    Extract text from a PDF while preserving page boundaries.
    """

    document = fitz.open(pdf_path)

    pages = []

    try:
        for page_number, page in enumerate(document, start=1):

            text = page.get_text("text").strip()

            if text:
                pages.append(
                    DocumentPage(
                        page_number=page_number,
                        text=text
                    )
                )

    finally:
        document.close()

    return ResearchDocument(
        source=pdf_path,
        pages=pages
    )

In [18]:
from google.colab import files

uploaded = files.upload()

Saving ravisankarmanem_resume_wellfound.pdf to ravisankarmanem_resume_wellfound.pdf


In [19]:
import shutil

filename = next(iter(uploaded.keys()))

pdf_path = PAPERS_DIR / filename

shutil.move(filename, pdf_path)

print("Paper saved to:")
print(pdf_path)

Paper saved to:
/content/Python_AI_engine/data/papers/ravisankarmanem_resume_wellfound.pdf


In [20]:
document = extract_pdf(str(pdf_path))

print("Source:", document.source)
print("Pages extracted:", document.page_count)
print("Characters extracted:", len(document.full_text))

Source: /content/Python_AI_engine/data/papers/ravisankarmanem_resume_wellfound.pdf
Pages extracted: 1
Characters extracted: 5588


In [21]:
for page in document.pages[:10]:
    print(
        f"Page {page.page_number}: "
        f"{len(page.text):,} characters"
    )

print(document.full_text[:5000])

Page 1: 5,588 characters
RAVI SANKAR MANEM
Eluru, Andhra Pradesh, India — +91-9347269455
manemravisankar28@gmail.com — GitHub — LinkedIn — LeetCode
Summary
Final-year Computer Science engineering student (B.Tech, expected May 2027) with backend engineering experience focused
on correctness under concurrency and financial-domain logic: idempotent payment processing, deadlock-free wallet transfers,
and third-party payment- gateway integration. Builds on the enterprise Java stack (Spring Boot, Spring Security, Hibernate,
REST/SOAP, microservices) with real CI/CD and production deployment experience (GitHub Actions, Docker, containerized
cloud hosting), and has worked at the JVM level itself – building a bytecode-instrumentation agent and debugging its internals.
Comfortable working in large or unfamiliar codebases, evidenced by a fix accepted into Graylog’s open-source Java codebase,
and communicates technical work clearly across teams.
Technical Skills
Languages: Java, Python, C, SQL, Ja

In [22]:
from dataclasses import dataclass


@dataclass
class DocumentChunk:
    chunk_id: int
    paper: str
    page_start: int
    page_end: int
    text: str
    token_count: int

In [23]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

print("Tokenizer loaded.")

Tokenizer loaded.


In [24]:
def tokenize(text: str) -> list[int]:
    return tokenizer.encode(text)


def detokenize(tokens: list[int]) -> str:
    return tokenizer.decode(tokens)

In [25]:
sample = "Graph neural networks are useful for modeling relationships between entities."

tokens = tokenize(sample)

print("Text:", sample)
print("Token count:", len(tokens))
print("Tokens:", tokens)
print("Decoded:", detokenize(tokens))

Text: Graph neural networks are useful for modeling relationships between entities.
Token count: 11
Tokens: [11461, 30828, 14488, 527, 5505, 369, 34579, 12135, 1990, 15086, 13]
Decoded: Graph neural networks are useful for modeling relationships between entities.


In [26]:
def create_chunks(
    document: ResearchDocument,
    chunk_size: int = 400,
    overlap: int = 80
) -> list[DocumentChunk]:

    if overlap >= chunk_size:
        raise ValueError(
            "overlap must be smaller than chunk_size"
        )

    chunks = []
    chunk_id = 0

    for page in document.pages:

        tokens = tokenize(page.text)

        start = 0

        while start < len(tokens):

            end = min(
                start + chunk_size,
                len(tokens)
            )

            chunk_tokens = tokens[start:end]

            text = detokenize(chunk_tokens).strip()

            if text:
                chunks.append(
                    DocumentChunk(
                        chunk_id=chunk_id,
                        paper=document.source,
                        page_start=page.page_number,
                        page_end=page.page_number,
                        text=text,
                        token_count=len(chunk_tokens)
                    )
                )

                chunk_id += 1

            # Move forward while preserving overlap
            start += chunk_size - overlap

    return chunks

In [27]:
chunks = create_chunks(
    document,
    chunk_size=400,
    overlap=80
)

print("Total chunks:", len(chunks))

Total chunks: 4


In [28]:
for chunk in chunks[:5]:
    print("=" * 80)
    print("Chunk ID:", chunk.chunk_id)
    print("Page:", chunk.page_start)
    print("Tokens:", chunk.token_count)
    print(chunk.text[:1000])

Chunk ID: 0
Page: 1
Tokens: 400
RAVI SANKAR MANEM
Eluru, Andhra Pradesh, India — +91-9347269455
manemravisankar28@gmail.com — GitHub — LinkedIn — LeetCode
Summary
Final-year Computer Science engineering student (B.Tech, expected May 2027) with backend engineering experience focused
on correctness under concurrency and financial-domain logic: idempotent payment processing, deadlock-free wallet transfers,
and third-party payment- gateway integration. Builds on the enterprise Java stack (Spring Boot, Spring Security, Hibernate,
REST/SOAP, microservices) with real CI/CD and production deployment experience (GitHub Actions, Docker, containerized
cloud hosting), and has worked at the JVM level itself – building a bytecode-instrumentation agent and debugging its internals.
Comfortable working in large or unfamiliar codebases, evidenced by a fix accepted into Graylog’s open-source Java codebase,
and communicates technical work clearly across teams.
Technical Skills
Languages: Java, Python, C, 

In [29]:
token_counts = [chunk.token_count for chunk in chunks]

print("Number of chunks:", len(token_counts))
print("Minimum tokens:", min(token_counts))
print("Maximum tokens:", max(token_counts))
print("Average tokens:", sum(token_counts) / len(token_counts))

Number of chunks: 4
Minimum tokens: 248
Maximum tokens: 400
Average tokens: 362.0


In [30]:
chunk = chunks[len(token_counts)-1]

print("Chunk ID:", chunk.chunk_id)
print("Paper:", chunk.paper)
print("Page:", chunk.page_start)
print("Token count:", chunk.token_count)
print()
print(chunk.text)

Chunk ID: 3
Paper: /content/Python_AI_engine/data/papers/ravisankarmanem_resume_wellfound.pdf
Page: 1
Token count: 248

-interception.
• Connected the agent’s output to SMTAP’s audit pipeline, streaming intercepted execution events so low-level JVM
instrumentation feeds directly into a higher-level compliance and audit system.
Open Source & Achievements
• Graylog (Java, open source) — Investigated an inefficiency where NodeMetricPeriodical kept invoking OSHI’s native
system-stats collector even after the corresponding configuration flag (disable native system stats collector) explicitly disabled
it; traced the cause to a missing conditional check in an unfamiliar production codebase, then submitted PR #26673, which
was reviewed and merged upstream.
• LeetCode: 200+ problems solved (125+ Medium, 15 Hard), reflecting sustained practice in algorithmic problem-solving.
Education
B.Tech in Computer Science and Engineering — RGUKT Nuzvid
Expected May 2027
CGPA: 8.53/10.0
Pre-University Cours

In [31]:
chunk_texts = [chunk.text for chunk in chunks]

chunk_embeddings = embed_texts(chunk_texts)

print("Chunks:", len(chunks))
print("Embeddings shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Chunks: 4
Embeddings shape: (4, 384)


In [32]:
print("First embedding:")
print(chunk_embeddings[0])

print("\nShape:")
print(chunk_embeddings[0].shape)

First embedding:
[-4.85236086e-02  4.16822955e-02  1.23238573e-02 -1.39668202e-02
  1.57807935e-02 -6.96598887e-02 -2.39972379e-02  4.52933088e-02
 -4.09192853e-02  2.18821364e-03  2.66673174e-02 -9.78010744e-02
  2.80930214e-02 -1.67049244e-02  7.56209940e-02 -1.08292745e-02
  1.08479394e-03 -9.88346338e-03  6.21853285e-02 -3.71253788e-02
  2.06017029e-02 -2.71381112e-03 -1.01546478e-02 -2.71953698e-02
 -7.59413280e-03  3.39598134e-02 -1.42991999e-02 -7.99328610e-02
 -2.88392846e-02 -1.49896204e-01  2.56408695e-02 -1.62625220e-02
  2.77399383e-02  2.90472843e-02 -8.14921316e-03  5.14636226e-02
 -2.62354128e-02 -5.83407879e-02 -2.23545581e-02 -1.37448506e-02
 -3.38089913e-02  2.62492243e-02  8.49612581e-04  2.43503060e-02
  1.39860418e-02 -8.73102024e-02  2.09260453e-02 -4.65238914e-02
 -5.94797395e-02 -9.41446051e-02 -7.97756482e-03 -4.38622236e-02
 -7.38345552e-03  1.89820603e-02  2.80150846e-02  1.52410020e-03
  4.74625528e-02  8.64250734e-02 -3.37953381e-02  4.92465720e-02
  2.5966

In [33]:
import faiss
import numpy as np

In [34]:
embedding_dimension = chunk_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(
    np.asarray(chunk_embeddings, dtype="float32")
)

print("FAISS index created.")
print("Vectors indexed:", faiss_index.ntotal)

FAISS index created.
Vectors indexed: 4


In [35]:
def semantic_search(
    query: str,
    top_k: int = 5
):
    """
    Retrieve the most semantically similar document chunks.
    """

    query_embedding = embed_text(query)

    query_vector = np.asarray(
        [query_embedding],
        dtype="float32"
    )

    scores, indices = faiss_index.search(
        query_vector,
        top_k
    )

    results = []

    for score, index in zip(scores[0], indices[0]):

        if index == -1:
            continue

        chunk = chunks[index]

        results.append({
            "chunk_id": chunk.chunk_id,
            "paper": chunk.paper,
            "page": chunk.page_start,
            "score": float(score),
            "text": chunk.text
        })

    return results

In [36]:
results = semantic_search(
    "How are graph neural networks used for blockchain wallet risk prediction?",
    top_k=5
)

In [37]:
for result in results:
    print("=" * 80)
    print("Chunk:", result["chunk_id"])
    print("Page:", result["page"])
    print("Score:", result["score"])
    print(result["text"][:1000])

Chunk: 1
Page: 1
Score: 0.6358395218849182
DevOps & Cloud: Docker, Kubernetes, AWS EC2, GitHub Actions (CI/CD), Maven, Git, Linux, Bash, PowerShell
Professional Skills: Team Collaboration — Technical Communication — Cross-functional Coordination
Projects
FinCore — Secure Payment Processing Platform (Fintech)
2026
[GitHub]
Java 21, Spring Boot, PostgreSQL, Razorpay API, Docker, GitHub Actions
• Architected a wallet-to-wallet payment engine that validates balance, currency, and wallet state with JWT/RBAC authoriza-
tion before allowing any transfer, so unauthorized or malformed requests never reach the ledger.
• Diagnosed a deadlock in concurrent wallet transfers by tracing how two opposing transfers acquired PostgreSQL locks in
different orders under load, then eliminated it by enforcing a deterministic wallet-ID locking sequence.
• Integrated the Razorpay payment gateway (Orders API, Checkout, HMAC-SHA256 signature verification), deliberately
separating order creation from wallet credi

In [38]:
query1 = "What dataset was used for blockchain wallet risk prediction?"

query2 = "What model architecture was used?"

In [39]:
for query in [query1, query2]:

    print("\n" + "#" * 100)
    print("QUERY:", query)

    results = semantic_search(query, top_k=3)

    for result in results:
        print("-" * 80)
        print(
            f"Chunk={result['chunk_id']} "
            f"Page={result['page']} "
            f"Score={result['score']:.4f}"
        )
        print(result["text"][:500])


####################################################################################################
QUERY: What dataset was used for blockchain wallet risk prediction?
--------------------------------------------------------------------------------
Chunk=1 Page=1 Score=0.6746
DevOps & Cloud: Docker, Kubernetes, AWS EC2, GitHub Actions (CI/CD), Maven, Git, Linux, Bash, PowerShell
Professional Skills: Team Collaboration — Technical Communication — Cross-functional Coordination
Projects
FinCore — Secure Payment Processing Platform (Fintech)
2026
[GitHub]
Java 21, Spring Boot, PostgreSQL, Razorpay API, Docker, GitHub Actions
• Architected a wallet-to-wallet payment engine that validates balance, currency, and wallet state with JWT/RBAC authoriza-
tion before allowing any
--------------------------------------------------------------------------------
Chunk=2 Page=1 Score=0.6371
Boot, PostgreSQL, Docker, Hash Chaining
• Delivered four independently deployable microservices (API Gateway, A

In [40]:
from rank_bm25 import BM25Okapi
import re

In [41]:
def tokenize_for_bm25(text: str) -> list[str]:
    """
    Simple lexical tokenizer for BM25.
    Converts text to lowercase and extracts word-like tokens.
    """
    return re.findall(r"\b\w+\b", text.lower())

In [42]:
sample = "Graph Neural Networks (GNNs) are used for wallet-risk prediction."

print(tokenize_for_bm25(sample))

['graph', 'neural', 'networks', 'gnns', 'are', 'used', 'for', 'wallet', 'risk', 'prediction']


In [43]:
bm25_corpus = [
    tokenize_for_bm25(chunk.text)
    for chunk in chunks
]

bm25_index = BM25Okapi(bm25_corpus)

print("BM25 index created.")
print("Documents indexed:", len(bm25_corpus))

BM25 index created.
Documents indexed: 4


In [44]:
def bm25_search(
    query: str,
    top_k: int = 5
):
    """
    Retrieve chunks using lexical BM25 ranking.
    """

    query_tokens = tokenize_for_bm25(query)

    scores = bm25_index.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in top_indices:

        chunk = chunks[index]

        results.append({
            "chunk_id": chunk.chunk_id,
            "paper": chunk.paper,
            "page": chunk.page_start,
            "score": float(scores[index]),
            "text": chunk.text
        })

    return results

In [45]:
results = bm25_search(
    "Graph Neural Network blockchain wallet risk prediction",
    top_k=5
)

In [46]:
for result in results:
    print("=" * 80)
    print("Chunk:", result["chunk_id"])
    print("Page:", result["page"])
    print("BM25 Score:", result["score"])
    print(result["text"][:700])

Chunk: 3
Page: 1
BM25 Score: 0.0
-interception.
• Connected the agent’s output to SMTAP’s audit pipeline, streaming intercepted execution events so low-level JVM
instrumentation feeds directly into a higher-level compliance and audit system.
Open Source & Achievements
• Graylog (Java, open source) — Investigated an inefficiency where NodeMetricPeriodical kept invoking OSHI’s native
system-stats collector even after the corresponding configuration flag (disable native system stats collector) explicitly disabled
it; traced the cause to a missing conditional check in an unfamiliar production codebase, then submitted PR #26673, which
was reviewed and merged upstream.
• LeetCode: 200+ problems solved (125+ Medium, 15 Hard), refl
Chunk: 2
Page: 1
BM25 Score: 0.0
Boot, PostgreSQL, Docker, Hash Chaining
• Delivered four independently deployable microservices (API Gateway, Auth, Audit, Security), each with a narrow responsibility,
so a change to one service does not require redeploying the othe

In [47]:
query = "What model was used for blockchain wallet risk prediction?"

In [48]:
dense_results = semantic_search(
    query,
    top_k=5
)

print("FAISS RESULTS")

for result in dense_results:
    print(
        f"Chunk={result['chunk_id']} "
        f"Page={result['page']} "
        f"Score={result['score']:.4f}"
    )


keyword_results = bm25_search(
    query,
    top_k=5
)

print("\nBM25 RESULTS")

for result in keyword_results:
    print(
        f"Chunk={result['chunk_id']} "
        f"Page={result['page']} "
        f"Score={result['score']:.4f}"
    )

FAISS RESULTS
Chunk=1 Page=1 Score=0.6882
Chunk=2 Page=1 Score=0.6437
Chunk=0 Page=1 Score=0.6190
Chunk=3 Page=1 Score=0.6155

BM25 RESULTS
Chunk=2 Page=1 Score=1.8974
Chunk=3 Page=1 Score=1.0117
Chunk=1 Page=1 Score=0.0000
Chunk=0 Page=1 Score=0.0000


In [49]:
def reciprocal_rank_fusion(
    result_lists: list[list[dict]],
    k: int = 60,
    top_k: int = 5
):
    """
    Combine ranked retrieval results using
    Reciprocal Rank Fusion (RRF).
    """

    fused_scores = {}
    result_lookup = {}

    for results in result_lists:

        for rank, result in enumerate(results, start=1):

            chunk_id = result["chunk_id"]

            fused_scores[chunk_id] = (
                fused_scores.get(chunk_id, 0.0)
                + 1.0 / (k + rank)
            )

            result_lookup[chunk_id] = result

    ranked_chunks = sorted(
        fused_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

    final_results = []

    for chunk_id, fusion_score in ranked_chunks[:top_k]:

        result = result_lookup[chunk_id].copy()

        result["fusion_score"] = fusion_score

        final_results.append(result)

    return final_results

In [50]:
def hybrid_search(
    query: str,
    top_k: int = 5,
    candidate_k: int = 10
):
    """
    Combine dense FAISS retrieval and BM25
    retrieval using Reciprocal Rank Fusion.
    """

    dense_results = semantic_search(
        query,
        top_k=candidate_k
    )

    keyword_results = bm25_search(
        query,
        top_k=candidate_k
    )

    return reciprocal_rank_fusion(
        [dense_results, keyword_results],
        top_k=top_k
    )

In [51]:
query = "What model architecture was used for blockchain wallet risk prediction?"

results = hybrid_search(
    query,
    top_k=5
)

for result in results:

    print("=" * 80)

    print(
        f"Chunk: {result['chunk_id']} | "
        f"Page: {result['page']} | "
        f"Fusion: {result['fusion_score']:.5f}"
    )

    print(result["text"][:1000])

Chunk: 2 | Page: 1 | Fusion: 0.03252
Boot, PostgreSQL, Docker, Hash Chaining
• Delivered four independently deployable microservices (API Gateway, Auth, Audit, Security), each with a narrow responsibility,
so a change to one service does not require redeploying the others.
• Protected historical audit records from tampering by chaining each event to the one before it with a SHA-256 hash, so any
retroactive edit breaks the chain on verification – a property auditors specifically check for.
• Resolved a slow tenant-scoped query by adding a composite PostgreSQL index on (tenant id, timestamp), cutting execution
time from a full table scan down to 0.05ms.
• Validated the API Gateway’s behavior under load with Grafana k6 at 1,000 virtual users, sustaining 316 requests/sec at 95%
success inside a Docker-containerized deployment, and used the result to locate the system’s concurrency ceiling.
Java Runtime Security Agent (JRSA)
2026
[GitHub]
Java, ByteBuddy, JVM Instrumentation API
• Engineere

In [52]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker loaded.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded.


In [53]:
query = "What model architecture was used for blockchain wallet risk prediction?"

candidates = hybrid_search(
    query=query,
    top_k=10,
    candidate_k=10
)

print("Candidates retrieved:", len(candidates))

print("\nFirst candidate keys:")
print(candidates[0].keys())

Candidates retrieved: 4

First candidate keys:
dict_keys(['chunk_id', 'paper', 'page', 'score', 'text', 'fusion_score'])


In [54]:
test_pair = [
    (
        query,
        candidates[0]["text"]
    )
]

score = reranker.predict(test_pair)

print("Reranker score:", float(score[0]))

Reranker score: -11.051819801330566


In [55]:
def rerank_results(
    query: str,
    candidates: list[dict],
    top_k: int = 5
):
    if not candidates:
        return []

    pairs = [
        (query, candidate["text"])
        for candidate in candidates
    ]

    scores = reranker.predict(pairs)

    reranked = []

    for candidate, score in zip(candidates, scores):
        result = candidate.copy()
        result["rerank_score"] = float(score)
        reranked.append(result)

    reranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:top_k]

In [56]:
reranked_results = rerank_results(
    query=query,
    candidates=candidates,
    top_k=5
)

for result in reranked_results:
    print("=" * 80)
    print(
        f"Chunk: {result['chunk_id']} | "
        f"Page: {result['page']} | "
        f"RRF: {result['fusion_score']:.5f} | "
        f"Rerank: {result['rerank_score']:.5f}"
    )
    print(result["text"][:700])

Chunk: 1 | Page: 1 | RRF: 0.03227 | Rerank: -8.49677
DevOps & Cloud: Docker, Kubernetes, AWS EC2, GitHub Actions (CI/CD), Maven, Git, Linux, Bash, PowerShell
Professional Skills: Team Collaboration — Technical Communication — Cross-functional Coordination
Projects
FinCore — Secure Payment Processing Platform (Fintech)
2026
[GitHub]
Java 21, Spring Boot, PostgreSQL, Razorpay API, Docker, GitHub Actions
• Architected a wallet-to-wallet payment engine that validates balance, currency, and wallet state with JWT/RBAC authoriza-
tion before allowing any transfer, so unauthorized or malformed requests never reach the ledger.
• Diagnosed a deadlock in concurrent wallet transfers by tracing how two opposing transfers acquired PostgreSQL locks in
differ
Chunk: 0 | Page: 1 | RRF: 0.03125 | Rerank: -10.25798
RAVI SANKAR MANEM
Eluru, Andhra Pradesh, India — +91-9347269455
manemravisankar28@gmail.com — GitHub — LinkedIn — LeetCode
Summary
Final-year Computer Science engineering student (B.Tech, expe

In [57]:
def build_context(results: list[dict]) -> str:
    """
    Build a context block from retrieved evidence.
    Each chunk retains its source page.
    """

    context_parts = []

    for i, result in enumerate(results, start=1):

        context_parts.append(
            f"""[Source {i}]
Paper: {result['paper']}
Page: {result['page']}
Chunk ID: {result['chunk_id']}

{result['text']}
"""
        )

    return "\n\n".join(context_parts)

In [58]:
def retrieve(
    query: str,
    candidate_k: int = 10,
    top_k: int = 5
):
    """
    Full retrieval pipeline:

    FAISS + BM25
        ↓
    Reciprocal Rank Fusion
        ↓
    Cross-encoder reranking
        ↓
    Top-K evidence
    """

    candidates = hybrid_search(
        query=query,
        top_k=candidate_k,
        candidate_k=candidate_k
    )

    return rerank_results(
        query=query,
        candidates=candidates,
        top_k=top_k
    )

In [59]:
query = "What model architecture was used for blockchain wallet risk prediction?"

results = retrieve(
    query=query,
    candidate_k=10,
    top_k=5
)

context = build_context(results)

print(context)

[Source 1]
Paper: /content/Python_AI_engine/data/papers/ravisankarmanem_resume_wellfound.pdf
Page: 1
Chunk ID: 1

DevOps & Cloud: Docker, Kubernetes, AWS EC2, GitHub Actions (CI/CD), Maven, Git, Linux, Bash, PowerShell
Professional Skills: Team Collaboration — Technical Communication — Cross-functional Coordination
Projects
FinCore — Secure Payment Processing Platform (Fintech)
2026
[GitHub]
Java 21, Spring Boot, PostgreSQL, Razorpay API, Docker, GitHub Actions
• Architected a wallet-to-wallet payment engine that validates balance, currency, and wallet state with JWT/RBAC authoriza-
tion before allowing any transfer, so unauthorized or malformed requests never reach the ledger.
• Diagnosed a deadlock in concurrent wallet transfers by tracing how two opposing transfers acquired PostgreSQL locks in
different orders under load, then eliminated it by enforcing a deterministic wallet-ID locking sequence.
• Integrated the Razorpay payment gateway (Orders API, Checkout, HMAC-SHA256 signature 

In [60]:
def build_rag_prompt(query: str, context: str) -> str:

    return f"""
You are a research assistant.

Answer the user's question using ONLY the research
evidence provided below.

Rules:
1. Do not invent facts.
2. Do not use information that is not supported by
   the provided evidence.
3. If the evidence is insufficient, say so explicitly.
4. Cite supporting evidence using [Source N].
5. Give a concise and technically accurate answer.

RESEARCH EVIDENCE
=================
{context}

QUESTION
========
{query}

ANSWER
======
"""

In [61]:
prompt = build_rag_prompt(
    query=query,
    context=context
)

answer = generate_answer(prompt)

print(answer)

The provided evidence does not contain any information about a model architecture used for blockchain wallet risk prediction. Therefore, I cannot answer the question based on the given sources.


In [62]:
def rag_query(
    query: str,
    candidate_k: int = 10,
    top_k: int = 5
):
    # Retrieve evidence
    results = retrieve(
        query=query,
        candidate_k=candidate_k,
        top_k=top_k
    )

    # Build context
    context = build_context(results)

    # Build grounded prompt
    prompt = build_rag_prompt(
        query=query,
        context=context
    )

    # Generate answer
    answer = generate_answer(prompt)

    return {
        "query": query,
        "answer": answer,
        "sources": results
    }

In [63]:
result = rag_query(
    "What model architecture was used for blockchain wallet risk prediction?"
)

print("ANSWER")
print("=" * 80)
print(result["answer"])

ANSWER
The provided evidence does not contain any information about a model architecture used for blockchain wallet risk prediction.


In [64]:
print("\nSOURCES")
print("=" * 80)

for source in result["sources"]:
    print(
        f"Page: {source['page']} | "
        f"Chunk: {source['chunk_id']} | "
        f"Rerank score: {source['rerank_score']:.4f}"
    )


SOURCES
Page: 1 | Chunk: 1 | Rerank score: -8.4968
Page: 1 | Chunk: 0 | Rerank score: -10.2580
Page: 1 | Chunk: 2 | Rerank score: -11.0518
Page: 1 | Chunk: 3 | Rerank score: -11.3362


In [65]:
result = rag_query(
    "What dataset was used for blockchain wallet risk prediction?"
)

print(result["answer"])

result = rag_query(
    "How was the GAT model trained and evaluated?"
)

print(result["answer"])

result = rag_query(
    "What was the total financial cost of developing this project?"
)

print(result["answer"])

The provided evidence does not contain any information about a dataset used for blockchain wallet risk prediction.
The provided evidence does not contain any information about a Graph Attention Network (GAT) model, its training process, or its evaluation methodology. Therefore, I cannot answer the question based on the given sources.
The provided evidence does not include any information about the monetary cost of developing the project, so the total financial cost cannot be determined from the given sources. [Source 1][Source 2][Source 3][Source 4]


# External Research Discovery

Discover related research papers from scholarly sources
such as OpenAlex and arXiv, rank them using semantic
similarity and cross-encoder reranking, and provide
direct links and metadata.

In [66]:
!pip install -q requests feedparser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 4.7 MB/s eta 0:00:00


In [67]:
import json
import logging
import requests
import feedparser
from urllib.parse import quote

# Single logger for the whole external-discovery pipeline.
# Using logging (not print) means Colab's cell output and any
# downstream log collector both get consistent, leveled messages.
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("research_discovery")


def build_paper_profile(document: ResearchDocument) -> str:
    """
    Create a compact representation of the research paper
    for research discovery.
    """

    pages = document.pages[:5]

    return "\n\n".join(
        page.text
        for page in pages
    )
paper_profile = build_paper_profile(document)

print(paper_profile[:5000])


In [68]:
def _fallback_queries(paper_profile: str, num_queries: int) -> list[str]:
    """
    Last-resort query generation used only if Groq is unreachable
    or returns something that cannot be parsed as JSON. This keeps
    discovery moving instead of failing the whole request.
    """

    words = [
        w.strip(".,:;()[]")
        for w in paper_profile[:2000].split()
        if len(w) > 5 and w.isalpha()
    ]

    seen = []
    for w in words:
        lw = w.lower()
        if lw not in seen:
            seen.append(lw)
        if len(seen) >= num_queries * 3:
            break

    queries = []
    for i in range(0, len(seen), 3):
        chunk = seen[i:i + 3]
        if chunk:
            queries.append(" ".join(chunk))
        if len(queries) >= num_queries:
            break

    return queries or ["research paper"]


def generate_research_queries(
    paper_profile: str,
    num_queries: int = 3
) -> list[str]:

    prompt = f"""
You are a research discovery assistant.

Analyze the research paper below and generate
{num_queries} precise scholarly search queries
for finding closely related research papers.

Focus on:
- research problem
- technical methodology
- algorithms/models
- application domain
- important technical terminology

Avoid:
- generic queries
- university names
- author names
- administrative information

Return ONLY valid JSON:

{{
    "queries": [
        "query 1",
        "query 2",
        "query 3"
    ]
}}

Research paper:
----------------
{paper_profile[:8000]}
----------------
"""

    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            max_tokens=300
        )

        content = response.choices[0].message.content.strip()

        # Handle accidental Markdown code fences
        if content.startswith("```"):
            content = content.replace("```json", "")
            content = content.replace("```", "")
            content = content.strip()

        data = json.loads(content)
        queries = data["queries"]

        if not queries:
            raise ValueError("Groq returned an empty query list.")

        return queries

    except (json.JSONDecodeError, KeyError, ValueError, TypeError) as exc:
        logger.warning(
            "Could not parse Groq query-generation output (%s). "
            "Falling back to keyword-based queries.",
            exc
        )
        return _fallback_queries(paper_profile, num_queries)

    except Exception as exc:
        # Groq itself unreachable/rate-limited/etc. Do not let this
        # take down the whole discovery pipeline.
        logger.warning(
            "Groq query generation failed (%s). "
            "Falling back to keyword-based queries.",
            exc
        )
        return _fallback_queries(paper_profile, num_queries)


In [69]:
research_queries = generate_research_queries(
    paper_profile
)

print("Generated research queries:\n")

for i, query in enumerate(research_queries, start=1):

    print(f"{i}. {query}")

Generated research queries:

1. deterministic lock ordering for deadlock prevention in concurrent financial wallet transfers using PostgreSQL row-level locks
2. hash chaining techniques for tamper-evident audit trails in multi-tenant microservice architectures
3. bytecode instrumentation with ByteBuddy for runtime interception of Java Runtime.exec calls and prevention of recursive instrumentation loops


In [70]:
def extract_openalex_abstract(work: dict) -> str:
    """
    Reconstruct an abstract from OpenAlex's
    inverted-index representation.
    """

    inverted_index = work.get("abstract_inverted_index")

    if not inverted_index:
        return ""

    words = []

    for word, positions in inverted_index.items():
        for position in positions:
            words.append((position, word))

    words.sort(key=lambda x: x[0])

    return " ".join(
        word for _, word in words
    )

In [79]:
def search_openalex(
    query: str,
    max_results: int = 10
) -> list[dict]:

    url = "https://api.openalex.org/works"

    params = {
        "search": query,
        "per-page": max_results
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=15
        )
        response.raise_for_status()

    except requests.exceptions.Timeout:
        logger.warning("OpenAlex timed out for query: %s", query)
        return []

    except requests.exceptions.ConnectionError:
        logger.warning(
            "OpenAlex is unreachable right now for query: %s", query
        )
        return []

    except requests.exceptions.HTTPError:
        logger.warning(
            "OpenAlex returned HTTP %s for query: %s",
            response.status_code, query
        )
        return []

    except requests.exceptions.RequestException as exc:
        logger.warning(
            "OpenAlex request failed for query '%s': %s", query, exc
        )
        return []

    try:
        data = response.json()

        papers = []

        for work in data.get("results", []):

            primary_location = work.get("primary_location") or {}

            papers.append({
                "title": work.get("display_name", ""),
                "authors": [
                    author["author"]["display_name"]
                    for author in work.get("authorships", [])
                    if author.get("author")
                ],
                "year": work.get("publication_year"),
                "abstract": extract_openalex_abstract(work),
                "doi": work.get("doi"),
                "url": primary_location.get("landing_page_url"),
                "source": "OpenAlex"
            })

        return papers

    except (ValueError, KeyError, TypeError) as exc:
        # Malformed/unexpected JSON shape - do not crash discovery.
        logger.warning(
            "Could not parse OpenAlex response for query '%s': %s",
            query, exc
        )
        return []


In [80]:
openalex_results = search_openalex(
    research_queries[0],
    max_results=5
)

print("OpenAlex results:", len(openalex_results))

for i, paper in enumerate(openalex_results, start=1):

    print("\n" + "=" * 80)
    print(f"#{i}")
    print("Title:", paper["title"])
    print("Authors:", ", ".join(paper["authors"][:5]))
    print("Year:", paper["year"])
    print("URL:", paper["url"])

OpenAlex search skipped for query: deterministic lock ordering for deadlock prevention in concurrent financial wallet transfers using PostgreSQL row-level locks
OpenAlex returned HTTP 503
OpenAlex results: 0


In [81]:
def search_arxiv(
    query: str,
    max_results: int = 10
) -> list[dict]:

    encoded_query = quote(query)

    url = (
        "https://export.arxiv.org/api/query"
        f"?search_query=all:{encoded_query}"
        f"&start=0"
        f"&max_results={max_results}"
        f"&sortBy=relevance"
        f"&sortOrder=descending"
    )

    try:
        response = requests.get(
            url,
            timeout=15
        )
        response.raise_for_status()

    except requests.exceptions.Timeout:
        logger.warning("arXiv timed out for query: %s", query)
        return []

    except requests.exceptions.ConnectionError:
        logger.warning(
            "arXiv is unreachable right now for query: %s", query
        )
        return []

    except requests.exceptions.HTTPError:
        logger.warning(
            "arXiv returned HTTP %s for query: %s",
            response.status_code, query
        )
        return []

    except requests.exceptions.RequestException as exc:
        logger.warning(
            "arXiv request failed for query '%s': %s", query, exc
        )
        return []

    try:
        feed = feedparser.parse(response.text)

        papers = []

        for entry in feed.entries:

            papers.append({
                "title": entry.get("title", "").replace("\n", " ").strip(),
                "authors": [
                    author.name
                    for author in entry.get("authors", [])
                ],
                "year": entry.get("published", "")[:4],
                "abstract": entry.get("summary", "").strip(),
                "doi": None,
                "url": entry.get("link"),
                "source": "arXiv"
            })

        return papers

    except Exception as exc:
        logger.warning(
            "Could not parse arXiv response for query '%s': %s",
            query, exc
        )
        return []


In [82]:
arxiv_results = search_arxiv(
    research_queries[0],
    max_results=5
)

print("arXiv results:", len(arxiv_results))

for i, paper in enumerate(arxiv_results, start=1):

    print("\n" + "=" * 80)
    print(f"#{i}")
    print("Title:", paper["title"])
    print("Authors:", ", ".join(paper["authors"][:5]))
    print("Year:", paper["year"])
    print("URL:", paper["url"])

arXiv results: 5

#1
Title: A Type System for Unstructured Locking that Guarantees Deadlock Freedom without Imposing a Lock Ordering
Authors: Prodromos Gerakios, Nikolaos Papaspyrou, Konstantinos Sagonas
Year: 2011
URL: https://arxiv.org/abs/1110.4160v1

#2
Title: No Cords Attached: Coordination-Free Concurrent Lock-Free Queues
Authors: Yusuf Motiwala
Year: 2025
URL: https://arxiv.org/abs/2511.09410v1

#3
Title: Phase-locking at low-level of quanta
Authors: G. H. Hovsepyan, A. R. Shahinyan, Lock Yue Chew, G. Yu. Kryuchkyan
Year: 2015
URL: https://arxiv.org/abs/1510.00670v1

#4
Title: Beyond Per-Thread Lock Sets: Multi-Thread Critical Sections and Dynamic Deadlock Prediction
Authors: Martin Sulzmann
Year: 2025
URL: https://arxiv.org/abs/2512.23552v2

#5
Title: Phase Locking between Two All-Optical Quantum Memories
Authors: Fumiya Okamoto, Mamoru Endo, Mikihisa Matsuyama, Yuya Ishizuka, Yang Liu
Year: 2020
URL: https://arxiv.org/abs/2009.06811v1


In [ ]:
def search_semantic_scholar(
    query: str,
    max_results: int = 10
) -> list[dict]:
    """
    Semantic Scholar's Graph API is free and requires no API key.
    Used as an additional/fallback source alongside OpenAlex and arXiv.
    """

    url = "https://api.semanticscholar.org/graph/v1/paper/search"

    params = {
        "query": query,
        "limit": max_results,
        "fields": "title,abstract,year,authors,externalIds,url"
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=15
        )
        response.raise_for_status()

    except requests.exceptions.Timeout:
        logger.warning("Semantic Scholar timed out for query: %s", query)
        return []

    except requests.exceptions.ConnectionError:
        logger.warning(
            "Semantic Scholar is unreachable right now for query: %s", query
        )
        return []

    except requests.exceptions.HTTPError:
        logger.warning(
            "Semantic Scholar returned HTTP %s for query: %s",
            response.status_code, query
        )
        return []

    except requests.exceptions.RequestException as exc:
        logger.warning(
            "Semantic Scholar request failed for query '%s': %s", query, exc
        )
        return []

    try:
        data = response.json()

        papers = []

        for work in data.get("data", []):

            external_ids = work.get("externalIds") or {}

            papers.append({
                "title": work.get("title", "") or "",
                "authors": [
                    author.get("name", "")
                    for author in work.get("authors", [])
                    if author.get("name")
                ],
                "year": work.get("year"),
                "abstract": work.get("abstract") or "",
                "doi": external_ids.get("DOI"),
                "url": work.get("url"),
                "source": "Semantic Scholar"
            })

        return papers

    except (ValueError, KeyError, TypeError) as exc:
        logger.warning(
            "Could not parse Semantic Scholar response for query '%s': %s",
            query, exc
        )
        return []


def _clean_crossref_abstract(raw_abstract) -> str:
    """Crossref abstracts come wrapped in JATS XML tags; strip them."""

    if not raw_abstract:
        return ""

    import re
    text = re.sub(r"<[^>]+>", " ", raw_abstract)
    return re.sub(r"\s+", " ", text).strip()


def search_crossref(
    query: str,
    max_results: int = 10
) -> list[dict]:
    """
    Crossref is a free, key-free metadata source for scholarly works.
    Many entries lack abstracts; those are simply filtered out later
    by the existing "usable abstract" check.
    """

    url = "https://api.crossref.org/works"

    params = {
        "query": query,
        "rows": max_results
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=15
        )
        response.raise_for_status()

    except requests.exceptions.Timeout:
        logger.warning("Crossref timed out for query: %s", query)
        return []

    except requests.exceptions.ConnectionError:
        logger.warning(
            "Crossref is unreachable right now for query: %s", query
        )
        return []

    except requests.exceptions.HTTPError:
        logger.warning(
            "Crossref returned HTTP %s for query: %s",
            response.status_code, query
        )
        return []

    except requests.exceptions.RequestException as exc:
        logger.warning(
            "Crossref request failed for query '%s': %s", query, exc
        )
        return []

    try:
        data = response.json()

        papers = []

        for item in data.get("message", {}).get("items", []):

            titles = item.get("title") or []
            title = titles[0] if titles else ""

            published = (
                item.get("published-print")
                or item.get("published-online")
                or item.get("issued")
                or {}
            )
            date_parts = published.get("date-parts") or [[None]]
            year = date_parts[0][0] if date_parts and date_parts[0] else None

            authors = [
                " ".join(
                    part for part in [
                        author.get("given"), author.get("family")
                    ] if part
                )
                for author in item.get("author", [])
            ]

            papers.append({
                "title": title,
                "authors": authors,
                "year": year,
                "abstract": _clean_crossref_abstract(item.get("abstract")),
                "doi": item.get("DOI"),
                "url": item.get("URL"),
                "source": "Crossref"
            })

        return papers

    except (ValueError, KeyError, TypeError) as exc:
        logger.warning(
            "Could not parse Crossref response for query '%s': %s",
            query, exc
        )
        return []


In [83]:
# All providers are free/open and require no API key.
# Each is called independently and defensively: a provider that is
# down, rate-limited, or unreachable simply contributes zero results
# for that query instead of stopping discovery for the other providers
# or the other queries.
DISCOVERY_PROVIDERS = [
    ("OpenAlex", search_openalex),
    ("arXiv", search_arxiv),
    ("Semantic Scholar", search_semantic_scholar),
    ("Crossref", search_crossref),
]


def discover_research(
    queries: list[str],
    results_per_query: int = 5
) -> list[dict]:

    all_papers = []

    for query in queries:

        print(f"\nSearching: {query}")

        for provider_name, provider_fn in DISCOVERY_PROVIDERS:

            try:
                results = provider_fn(
                    query,
                    max_results=results_per_query
                )

            except Exception as exc:
                # Belt-and-braces: even though each provider function
                # already catches its own network/parsing errors, this
                # guarantees a bug in one provider can never take down
                # the rest of discovery.
                logger.warning(
                    "%s raised an unexpected error for query '%s': %s",
                    provider_name, query, exc
                )
                results = []

            if results:
                print(f"  {provider_name}: {len(results)} result(s)")
            else:
                print(f"  {provider_name}: 0 result(s) (skipped/unreachable)")

            all_papers.extend(results)

    return all_papers


In [84]:
external_candidates = discover_research(
    research_queries,
    results_per_query=5
)

print(
    "\nTotal external candidates:",
    len(external_candidates)
)


Searching: deterministic lock ordering for deadlock prevention in concurrent financial wallet transfers using PostgreSQL row-level locks
OpenAlex search skipped for query: deterministic lock ordering for deadlock prevention in concurrent financial wallet transfers using PostgreSQL row-level locks
OpenAlex returned HTTP 503

Searching: hash chaining techniques for tamper-evident audit trails in multi-tenant microservice architectures
OpenAlex search skipped for query: hash chaining techniques for tamper-evident audit trails in multi-tenant microservice architectures
OpenAlex returned HTTP 503

Searching: bytecode instrumentation with ByteBuddy for runtime interception of Java Runtime.exec calls and prevention of recursive instrumentation loops

Total external candidates: 15


In [85]:
import requests

url = "https://export.arxiv.org/api/query"

params = {
    "search_query": "all:ByteBuddy",
    "start": 0,
    "max_results": 5,
    "sortBy": "relevance",
    "sortOrder": "descending"
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.url)

200
https://export.arxiv.org/api/query?search_query=all%3AByteBuddy&start=0&max_results=5&sortBy=relevance&sortOrder=descending


In [86]:
def deduplicate_papers(
    papers: list[dict]
) -> list[dict]:

    seen_titles = set()
    unique_papers = []

    for paper in papers:

        title = paper["title"].lower().strip()

        if not title:
            continue

        if title in seen_titles:
            continue

        seen_titles.add(title)

        unique_papers.append(paper)

    return unique_papers


external_candidates = deduplicate_papers(
    external_candidates
)

print(
    "Unique external papers:",
    len(external_candidates)
)

Unique external papers: 15


In [87]:
for i, paper in enumerate(
    external_candidates,
    start=1
):

    print("\n" + "=" * 100)
    print(f"#{i}")
    print("Title:", paper["title"])
    print("Year:", paper["year"])
    print("Source:", paper["source"])
    print("URL:", paper["url"])


#1
Title: A Type System for Unstructured Locking that Guarantees Deadlock Freedom without Imposing a Lock Ordering
Year: 2011
Source: arXiv
URL: https://arxiv.org/abs/1110.4160v1

#2
Title: No Cords Attached: Coordination-Free Concurrent Lock-Free Queues
Year: 2025
Source: arXiv
URL: https://arxiv.org/abs/2511.09410v1

#3
Title: Phase-locking at low-level of quanta
Year: 2015
Source: arXiv
URL: https://arxiv.org/abs/1510.00670v1

#4
Title: Beyond Per-Thread Lock Sets: Multi-Thread Critical Sections and Dynamic Deadlock Prediction
Year: 2025
Source: arXiv
URL: https://arxiv.org/abs/2512.23552v2

#5
Title: Phase Locking between Two All-Optical Quantum Memories
Year: 2020
Source: arXiv
URL: https://arxiv.org/abs/2009.06811v1

#6
Title: Incentivizing Multi-Tenant Split Federated Learning for Foundation Models at the Network Edge
Year: 2025
Source: arXiv
URL: https://arxiv.org/abs/2503.04971v2

#7
Title: Constructions and bounds for separating hash families
Year: 2016
Source: arXiv
URL: ht

In [88]:
source_paper_text = paper_profile[:12000]

source_paper_embedding = embed_text(
    source_paper_text
)

print("Source embedding shape:", source_paper_embedding.shape)

Source embedding shape: (384,)


In [89]:
def build_external_paper_text(paper: dict) -> str:
    title = paper.get("title", "").strip()
    abstract = paper.get("abstract", "").strip()

    return (
        f"Title: {title}\n"
        f"Abstract: {abstract}"
    )


rankable_papers = [
    paper
    for paper in external_candidates
    if paper.get("title")
    and paper.get("abstract")
    and len(paper["abstract"].strip()) > 50
]

print("Papers with usable abstracts:", len(rankable_papers))

Papers with usable abstracts: 15


In [90]:
candidate_texts = [
    build_external_paper_text(paper)
    for paper in rankable_papers
]

candidate_embeddings = embed_texts(
    candidate_texts
)

print("Candidate embeddings:", candidate_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Candidate embeddings: (15, 384)


In [91]:
similarity_scores = (
    candidate_embeddings @ source_paper_embedding
)

for paper, score in zip(
    rankable_papers,
    similarity_scores
):
    paper["embedding_similarity"] = float(score)

rankable_papers.sort(
    key=lambda paper: paper["embedding_similarity"],
    reverse=True
)

In [92]:
for i, paper in enumerate(
    rankable_papers[:10],
    start=1
):
    print("\n" + "=" * 100)
    print(f"#{i}")
    print("Title:", paper["title"])
    print("Similarity:", round(
        paper["embedding_similarity"], 4
    ))
    print("Year:", paper["year"])
    print("Source:", paper["source"])
    print("URL:", paper["url"])


#1
Title: Audit Trails for Accountability in Large Language Models
Similarity: 0.6467
Year: 2026
Source: arXiv
URL: https://arxiv.org/abs/2601.20727v1

#2
Title: Incentivizing Multi-Tenant Split Federated Learning for Foundation Models at the Network Edge
Similarity: 0.6459
Year: 2025
Source: arXiv
URL: https://arxiv.org/abs/2503.04971v2

#3
Title: Beyond Per-Thread Lock Sets: Multi-Thread Critical Sections and Dynamic Deadlock Prediction
Similarity: 0.6384
Year: 2025
Source: arXiv
URL: https://arxiv.org/abs/2512.23552v2

#4
Title: Auditing Privacy in Multi-Tenant RAG under Account Collusion
Similarity: 0.6358
Year: 2026
Source: arXiv
URL: https://arxiv.org/abs/2605.19847v2

#5
Title: No Cords Attached: Coordination-Free Concurrent Lock-Free Queues
Similarity: 0.6345
Year: 2025
Source: arXiv
URL: https://arxiv.org/abs/2511.09410v1

#6
Title: A Type System for Unstructured Locking that Guarantees Deadlock Freedom without Imposing a Lock Ordering
Similarity: 0.6158
Year: 2011
Source: ar

In [93]:
def rerank_external_papers(
    source_text: str,
    papers: list[dict],
    candidate_k: int = 20,
    top_k: int = 10
):
    """
    Rerank externally discovered papers using
    a cross-encoder.
    """

    candidates = papers[:candidate_k]

    pairs = [
        (
            source_text,
            build_external_paper_text(paper)
        )
        for paper in candidates
    ]

    scores = reranker.predict(pairs)

    ranked = []

    for paper, score in zip(
        candidates,
        scores
    ):
        result = paper.copy()

        result["rerank_score"] = float(score)

        ranked.append(result)

    ranked.sort(
        key=lambda paper: paper["rerank_score"],
        reverse=True
    )

    return ranked[:top_k]

In [94]:
similar_papers = rerank_external_papers(
    source_text=source_paper_text,
    papers=rankable_papers,
    candidate_k=20,
    top_k=10
)

print(
    "Final similar papers:",
    len(similar_papers)
)

Final similar papers: 10


In [95]:
for i, paper in enumerate(
    similar_papers,
    start=1
):

    print("\n" + "=" * 100)

    print(f"#{i}")
    print("Title:", paper["title"])
    print(
        "Authors:",
        ", ".join(paper["authors"][:5])
    )
    print("Year:", paper["year"])
    print("Source:", paper["source"])

    print(
        "Embedding similarity:",
        round(
            paper["embedding_similarity"],
            4
        )
    )

    print(
        "Reranker score:",
        round(
            paper["rerank_score"],
            4
        )
    )

    print("URL:", paper["url"])


#1
Title: A Type System for Unstructured Locking that Guarantees Deadlock Freedom without Imposing a Lock Ordering
Authors: Prodromos Gerakios, Nikolaos Papaspyrou, Konstantinos Sagonas
Year: 2011
Source: arXiv
Embedding similarity: 0.6158
Reranker score: -10.0373
URL: https://arxiv.org/abs/1110.4160v1

#2
Title: Constructions and bounds for separating hash families
Authors: X. Niu, H. Cao
Year: 2016
Source: arXiv
Embedding similarity: 0.5891
Reranker score: -10.2873
URL: https://arxiv.org/abs/1611.03274v2

#3
Title: Phase-locking at low-level of quanta
Authors: G. H. Hovsepyan, A. R. Shahinyan, Lock Yue Chew, G. Yu. Kryuchkyan
Year: 2015
Source: arXiv
Embedding similarity: 0.5074
Reranker score: -10.4571
URL: https://arxiv.org/abs/1510.00670v1

#4
Title: Phase Locking between Two All-Optical Quantum Memories
Authors: Fumiya Okamoto, Mamoru Endo, Mikihisa Matsuyama, Yuya Ishizuka, Yang Liu
Year: 2020
Source: arXiv
Embedding similarity: 0.5315
Reranker score: -10.515
URL: https://arxiv

In [96]:
def explain_paper_relationship(
    source_paper: str,
    related_paper: dict
) -> str:

    prompt = f"""
You are a research analysis assistant.

Compare the source research paper with the related paper.

SOURCE PAPER
============
{source_paper[:4000]}

RELATED PAPER
=============
Title: {related_paper["title"]}

Authors:
{", ".join(related_paper.get("authors", [])[:10])}

Abstract:
{related_paper.get("abstract", "")[:2000]}

Analyze ONLY information supported by the provided text.

Explain:

1. Shared research problem
2. Shared methodology or algorithms
3. Shared application/domain
4. Important differences
5. Why the related paper is relevant to the source paper

Do not invent datasets, results, algorithms, or conclusions.

Return the analysis in this format:

Shared Problem:
...

Shared Methods:
...

Shared Domain:
...

Key Differences:
...

Why It Is Relevant:
...
"""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=600
    )

    return response.choices[0].message.content


In [ ]:
def explain_paper_relationships_batch(
    source_paper: str,
    related_papers: list[dict]
) -> list[dict]:
    """
    Token-efficient alternative to calling explain_paper_relationship()
    once per paper. The source paper text is sent to Groq exactly ONCE
    no matter how many related papers are being explained, and all
    explanations come back in a single JSON response.

    Falls back to per-paper explanations only if the batched call
    fails or returns malformed JSON, so relationship analysis never
    silently disappears.
    """

    if not related_papers:
        return []

    papers_block = "\n\n".join(
        f"[{i}] Title: {paper['title']}\n"
        f"Authors: {', '.join(paper.get('authors', [])[:6])}\n"
        f"Abstract: {paper.get('abstract', '')[:1200]}"
        for i, paper in enumerate(related_papers)
    )

    prompt = f"""
You are a research analysis assistant.

Compare the SOURCE paper below with each RELATED paper listed.
Analyze ONLY information supported by the provided text - do not
invent datasets, results, algorithms, or conclusions.

SOURCE PAPER
============
{source_paper[:4000]}

RELATED PAPERS
==============
{papers_block}

Return ONLY valid JSON in this exact shape:

{{
  "explanations": [
    {{
      "index": 0,
      "shared_problem": "...",
      "shared_methods": "...",
      "shared_domain": "...",
      "key_differences": "...",
      "why_relevant": "..."
    }}
  ]
}}

Include one object per related paper, in the same order, using the
same "index" values shown above.
"""

    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=1200
        )

        content = response.choices[0].message.content.strip()

        if content.startswith("```"):
            content = content.replace("```json", "").replace("```", "").strip()

        data = json.loads(content)
        explanations = {
            item["index"]: item
            for item in data["explanations"]
        }

        formatted = []
        for i in range(len(related_papers)):
            item = explanations.get(i)
            if not item:
                formatted.append("Relationship analysis unavailable.")
                continue

            formatted.append(
                f"Shared Problem:\n{item.get('shared_problem', '')}\n\n"
                f"Shared Methods:\n{item.get('shared_methods', '')}\n\n"
                f"Shared Domain:\n{item.get('shared_domain', '')}\n\n"
                f"Key Differences:\n{item.get('key_differences', '')}\n\n"
                f"Why It Is Relevant:\n{item.get('why_relevant', '')}"
            )

        return formatted

    except (json.JSONDecodeError, KeyError, ValueError, TypeError) as exc:
        logger.warning(
            "Batched relationship analysis returned unparsable JSON (%s). "
            "Falling back to one Groq call per paper.",
            exc
        )

    except Exception as exc:
        logger.warning(
            "Batched relationship analysis failed (%s). "
            "Falling back to one Groq call per paper.",
            exc
        )

    # Fallback: one call per paper (still correct, just less efficient).
    return [
        explain_paper_relationship(source_paper, paper)
        for paper in related_papers
    ]


In [97]:
top_paper = similar_papers[0]

explanation = explain_paper_relationship(
    source_paper=source_paper_text,
    related_paper=top_paper
)

print("RELATED PAPER")
print("=" * 100)
print(top_paper["title"])

print("\nANALYSIS")
print("=" * 100)
print(explanation)

RELATED PAPER
A Type System for Unstructured Locking that Guarantees Deadlock Freedom without Imposing a Lock Ordering

ANALYSIS
**Shared Problem:**  
Both the source document (the résumé/project description) and the related paper address the *deadlock problem* that arises in concurrent programs when multiple threads acquire shared resources (locks) in conflicting orders. The source work reports a real‑world deadlock that occurred during wallet‑to‑wallet transfers in a fintech payment engine and the need to guarantee that such deadlocks cannot happen. The related paper tackles the same fundamental issue by proposing a type system that *guarantees deadlock freedom* for programs that use unstructured (i.e., not pre‑ordered) lock acquisition.

**Shared Methods:**  
- **Analysis of lock acquisition patterns:**  
  - *Source:* The engineer traced the deadlock to two opposing transfers that acquired PostgreSQL row‑level locks in opposite orders, then eliminated it by enforcing a deterministi

In [98]:
for i, paper in enumerate(similar_papers[:3], start=1):

    print("\n" + "#" * 100)
    print(f"RELATED PAPER #{i}")
    print("#" * 100)

    print("Title:", paper["title"])
    print("Year:", paper["year"])
    print("Source:", paper["source"])
    print("URL:", paper["url"])

    explanation = explain_paper_relationship(
        source_paper=source_paper_text,
        related_paper=paper
    )

    print("\n" + explanation)


####################################################################################################
RELATED PAPER #1
####################################################################################################
Title: A Type System for Unstructured Locking that Guarantees Deadlock Freedom without Imposing a Lock Ordering
Year: 2011
Source: arXiv
URL: https://arxiv.org/abs/1110.4160v1

**Shared Problem:**  
Both works address the *deadlock* problem that arises in concurrent programs when multiple threads acquire shared resources (e.g., locks, database rows) in conflicting orders, leading to cyclic wait conditions that halt progress.

**Shared Methods:**  
- **Concurrency control** is central to both.  
- The source paper’s *FinCore* project resolves a deadlock by **enforcing a deterministic lock‑acquisition order** on wallet IDs, thereby eliminating cycles at runtime.  
- The related paper proposes a **type‑system‑based analysis** that statically guarantees deadlock freedom for

In [99]:
def discover_similar_research(
    document: ResearchDocument,
    num_queries: int = 3,
    results_per_query: int = 5,
    candidate_k: int = 20,
    top_k: int = 10,
    explain_top: int = 3
):
    """
    Complete external research discovery pipeline.

    PDF
      ↓
    Research profile
      ↓
    Groq query generation
      ↓
    OpenAlex + arXiv + Semantic Scholar + Crossref
      ↓
    Deduplication
      ↓
    BGE semantic ranking
      ↓
    Cross-encoder reranking
      ↓
    Groq relationship analysis (batched)
    """

    # --------------------------------------------------
    # 1. Build research profile
    # --------------------------------------------------

    profile = build_paper_profile(document)

    # --------------------------------------------------
    # 2. Generate scholarly queries
    #    (falls back to keyword queries if Groq is down)
    # --------------------------------------------------

    queries = generate_research_queries(
        profile,
        num_queries=num_queries
    )

    # --------------------------------------------------
    # 3. Search external research (each provider isolated)
    # --------------------------------------------------

    candidates = discover_research(
        queries,
        results_per_query=results_per_query
    )

    # --------------------------------------------------
    # 4. Deduplicate
    # --------------------------------------------------

    candidates = deduplicate_papers(
        candidates
    )

    # --------------------------------------------------
    # 5. Keep papers with abstracts
    # --------------------------------------------------

    rankable = [
        paper
        for paper in candidates
        if paper.get("title")
        and paper.get("abstract")
        and len(
            paper["abstract"].strip()
        ) > 50
    ]

    if not rankable:
        return {
            "queries": queries,
            "papers": []
        }

    # --------------------------------------------------
    # 6. Source embedding
    #    (6000 chars is enough signal for ranking and
    #    keeps embedding calls cheaper)
    # --------------------------------------------------

    source_embedding = embed_text(
        profile[:6000]
    )

    # --------------------------------------------------
    # 7. Candidate embeddings
    # --------------------------------------------------

    candidate_texts = [
        build_external_paper_text(paper)
        for paper in rankable
    ]

    candidate_embeddings = embed_texts(
        candidate_texts
    )

    # --------------------------------------------------
    # 8. Semantic similarity
    # --------------------------------------------------

    scores = (
        candidate_embeddings
        @ source_embedding
    )

    for paper, score in zip(
        rankable,
        scores
    ):
        paper["embedding_similarity"] = float(
            score
        )

    rankable.sort(
        key=lambda x: x["embedding_similarity"],
        reverse=True
    )

    # --------------------------------------------------
    # 9. Cross-encoder reranking
    # --------------------------------------------------

    final_papers = rerank_external_papers(
        source_text=profile[:6000],
        papers=rankable,
        candidate_k=candidate_k,
        top_k=top_k
    )

    # --------------------------------------------------
    # 10. Explain top papers - ONE batched Groq call
    #     instead of one call per paper.
    # --------------------------------------------------

    top_papers = final_papers[:explain_top]

    explanations = explain_paper_relationships_batch(
        source_paper=profile[:4000],
        related_papers=top_papers
    )

    for paper, explanation in zip(top_papers, explanations):
        paper["relationship_analysis"] = explanation

    return {
        "queries": queries,
        "papers": final_papers
    }


In [100]:
research_discovery = discover_similar_research(
    document
)


Searching: deterministic lock ordering for deadlock prevention in concurrent financial transaction processing systems
OpenAlex search skipped for query: deterministic lock ordering for deadlock prevention in concurrent financial transaction processing systems
OpenAlex returned HTTP 503

Searching: hash chaining techniques for tamper‑evident multi‑tenant audit logs in microservice architectures
OpenAlex search skipped for query: hash chaining techniques for tamper‑evident multi‑tenant audit logs in microservice architectures
OpenAlex returned HTTP 503

Searching: runtime bytecode instrumentation agents for monitoring Java Runtime.exec calls while avoiding recursive self‑interception
OpenAlex search skipped for query: runtime bytecode instrumentation agents for monitoring Java Runtime.exec calls while avoiding recursive self‑interception
OpenAlex returned HTTP 503


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [101]:
print("SEARCH QUERIES")
print("=" * 100)

for query in research_discovery["queries"]:
    print("-", query)

SEARCH QUERIES
- deterministic lock ordering for deadlock prevention in concurrent financial transaction processing systems
- hash chaining techniques for tamper‑evident multi‑tenant audit logs in microservice architectures
- runtime bytecode instrumentation agents for monitoring Java Runtime.exec calls while avoiding recursive self‑interception


In [102]:
print("\nRELATED RESEARCH")
print("=" * 100)

for i, paper in enumerate(
    research_discovery["papers"],
    start=1
):

    print(f"\n#{i}")
    print("Title:", paper["title"])
    print("Year:", paper["year"])
    print("Source:", paper["source"])

    print(
        "Embedding similarity:",
        round(
            paper["embedding_similarity"],
            4
        )
    )

    print(
        "Reranker score:",
        round(
            paper["rerank_score"],
            4
        )
    )

    print("Link:", paper["url"])


RELATED RESEARCH

#1
Title: A Type System for Unstructured Locking that Guarantees Deadlock Freedom without Imposing a Lock Ordering
Year: 2011
Source: arXiv
Embedding similarity: 0.6158
Reranker score: -10.0373
Link: https://arxiv.org/abs/1110.4160v1

#2
Title: Constructions and bounds for separating hash families
Year: 2016
Source: arXiv
Embedding similarity: 0.5891
Reranker score: -10.2873
Link: https://arxiv.org/abs/1611.03274v2

#3
Title: Introduction to probabilistic concurrent systems
Year: 2021
Source: arXiv
Embedding similarity: 0.5717
Reranker score: -10.3736
Link: https://arxiv.org/abs/2111.00507v6

#4
Title: Efficient and Expressive Bytecode-Level Instrumentation for Java Programs
Year: 2021
Source: arXiv
Embedding similarity: 0.7036
Reranker score: -10.563
Link: https://arxiv.org/abs/2106.01115v1

#5
Title: BISM: Bytecode-Level Instrumentation for Software Monitoring
Year: 2020
Source: arXiv
Embedding similarity: 0.6884
Reranker score: -10.6053
Link: https://arxiv.org/abs

In [103]:
print("\nRESEARCH RELATIONSHIPS")
print("=" * 100)

for paper in research_discovery["papers"][:3]:

    print("\n" + "-" * 100)
    print(paper["title"])
    print("-" * 100)

    print(
        paper["relationship_analysis"]
    )


RESEARCH RELATIONSHIPS

----------------------------------------------------------------------------------------------------
A Type System for Unstructured Locking that Guarantees Deadlock Freedom without Imposing a Lock Ordering
----------------------------------------------------------------------------------------------------
**Shared Problem:**  
Both works address *deadlock* in concurrent systems.  
- The source paper reports a real‑world deadlock that arose when two opposing wallet‑to‑wallet transfers acquired PostgreSQL locks in different orders, and describes how the author eliminated it.  
- The related paper tackles deadlock as a fundamental issue of cyclic resource acquisition between threads and proposes a type system that guarantees deadlock freedom.

**Shared Methods:**  
Both approaches involve *controlling the order in which locks are obtained* to prevent cycles.  
- In the source project the author solved the problem by enforcing a **deterministic wallet‑ID locking se

In [104]:
queries = [
    "What model architecture was used for blockchain wallet risk prediction?",
    "What dataset was used for blockchain wallet risk prediction?",
    "What features were used for the wallet risk prediction model?",
    "How was the model evaluated?"
]

for query in queries:

    print("\n" + "=" * 100)
    print("QUERY:", query)
    print("=" * 100)

    results = retrieve(
        query=query,
        candidate_k=10,
        top_k=5
    )

    for rank, result in enumerate(results, start=1):

        print(
            f"\nRank {rank}"
            f" | Chunk: {result['chunk_id']}"
            f" | Page: {result['page']}"
            f" | Score: {result.get('rerank_score', 'N/A')}"
        )

        print(result["text"][:700])


QUERY: What model architecture was used for blockchain wallet risk prediction?

Rank 1 | Chunk: 1 | Page: 1 | Score: -8.496768951416016
DevOps & Cloud: Docker, Kubernetes, AWS EC2, GitHub Actions (CI/CD), Maven, Git, Linux, Bash, PowerShell
Professional Skills: Team Collaboration — Technical Communication — Cross-functional Coordination
Projects
FinCore — Secure Payment Processing Platform (Fintech)
2026
[GitHub]
Java 21, Spring Boot, PostgreSQL, Razorpay API, Docker, GitHub Actions
• Architected a wallet-to-wallet payment engine that validates balance, currency, and wallet state with JWT/RBAC authoriza-
tion before allowing any transfer, so unauthorized or malformed requests never reach the ledger.
• Diagnosed a deadlock in concurrent wallet transfers by tracing how two opposing transfers acquired PostgreSQL locks in
differ

Rank 2 | Chunk: 0 | Page: 1 | Score: -10.257984161376953
RAVI SANKAR MANEM
Eluru, Andhra Pradesh, India — +91-9347269455
manemravisankar28@gmail.com — GitHub — L

In [105]:
import math


evaluation_dataset = [
    {
        "query": "What model architecture was used for blockchain wallet risk prediction?",
        "relevant_chunks": [24, 48]
    },

    # We will fill these after inspecting the results
    {
        "query": "What dataset was used for blockchain wallet risk prediction?",
        "relevant_chunks": []
    },

    {
        "query": "What features were used for the wallet risk prediction model?",
        "relevant_chunks": []
    },

    {
        "query": "How was the model evaluated?",
        "relevant_chunks": []
    }
]


def recall_at_k(results, relevant_chunks, k=5):

    retrieved_chunks = {
        result["chunk_id"]
        for result in results[:k]
    }

    relevant_chunks = set(relevant_chunks)

    if not relevant_chunks:
        return 0.0

    return len(
        retrieved_chunks & relevant_chunks
    ) / len(relevant_chunks)


def reciprocal_rank(results, relevant_chunks):

    relevant_chunks = set(relevant_chunks)

    for rank, result in enumerate(
        results,
        start=1
    ):

        if result["chunk_id"] in relevant_chunks:
            return 1.0 / rank

    return 0.0


def ndcg_at_k(results, relevant_chunks, k=5):

    relevant_chunks = set(relevant_chunks)

    dcg = 0.0

    for rank, result in enumerate(
        results[:k],
        start=1
    ):

        relevance = (
            1
            if result["chunk_id"] in relevant_chunks
            else 0
        )

        dcg += (
            relevance /
            math.log2(rank + 1)
        )

    ideal_relevant = min(
        len(relevant_chunks),
        k
    )

    if ideal_relevant == 0:
        return 0.0

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(
            1,
            ideal_relevant + 1
        )
    )

    return dcg / idcg


def evaluate_retrieval(
    evaluation_dataset,
    candidate_k=10,
    top_k=5
):

    recalls = []
    reciprocal_ranks = []
    ndcg_scores = []

    valid_queries = 0

    for item in evaluation_dataset:

        # Don't evaluate queries whose ground truth
        # has not yet been established.
        if not item["relevant_chunks"]:
            continue

        valid_queries += 1

        results = retrieve(
            query=item["query"],
            candidate_k=candidate_k,
            top_k=top_k
        )

        recalls.append(
            recall_at_k(
                results,
                item["relevant_chunks"],
                k=top_k
            )
        )

        reciprocal_ranks.append(
            reciprocal_rank(
                results,
                item["relevant_chunks"]
            )
        )

        ndcg_scores.append(
            ndcg_at_k(
                results,
                item["relevant_chunks"],
                k=top_k
            )
        )

    if valid_queries == 0:
        return {
            "Recall@5": 0.0,
            "MRR": 0.0,
            "NDCG@5": 0.0
        }

    return {
        "Recall@5": sum(recalls) / len(recalls),
        "MRR": sum(reciprocal_ranks) / len(reciprocal_ranks),
        "NDCG@5": sum(ndcg_scores) / len(ndcg_scores)
    }

In [106]:
metrics = evaluate_retrieval(
    evaluation_dataset
)

print("Retrieval Evaluation")
print("=" * 50)

for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

Retrieval Evaluation
Recall@5: 0.0000
MRR: 0.0000
NDCG@5: 0.0000


In [107]:
# ============================================================
# FASTAPI AI ENGINE
# Spring Boot Integration Layer
# ============================================================

# Install dependencies if not already installed
!pip install -q fastapi uvicorn python-multipart pyngrok


# ============================================================
# IMPORTS
# ============================================================

import os
import shutil
import threading
import time

from pathlib import Path

from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field


# ============================================================
# API CONFIGURATION
# ============================================================

AI_ENGINE_PORT = 8000

API_UPLOAD_DIR = Path("/content/research_api_uploads")
API_UPLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# FASTAPI APPLICATION
# ============================================================

app = FastAPI(
    title="Research Retrieval & Reasoning Engine",
    description=(
        "AI engine for research paper retrieval, "
        "RAG question answering, and external research discovery."
    ),
    version="1.0.0"
)


# ============================================================
# CORS
# ============================================================

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


# ============================================================
# REQUEST MODELS
# ============================================================

class RetrievalRequest(BaseModel):

    query: str = Field(
        ...,
        min_length=1,
        description="Research question"
    )

    candidate_k: int = Field(
        default=10,
        ge=1,
        le=50
    )

    # NOTE: kept as "topK" (camelCase) on purpose - this is the field
    # name the Spring Boot RetrievalRequest DTO sends over the wire.
    topK: int = Field(
        default=5,
        ge=1,
        le=20
    )


class DiscoveryRequest(BaseModel):

    num_queries: int = Field(
        default=3,
        ge=1,
        le=5
    )

    results_per_query: int = Field(
        default=5,
        ge=1,
        le=20
    )

    candidate_k: int = Field(
        default=20,
        ge=1,
        le=50
    )

    top_k: int = Field(
        default=10,
        ge=1,
        le=20
    )

    explain_top: int = Field(
        default=3,
        ge=0,
        le=10
    )


# ============================================================
# GLOBAL AI ENGINE STATE
# ============================================================

ENGINE_STATE = {
    "ready": False,
    "document_name": None,
    "document_source": None,
    "chunk_count": 0
}


# ============================================================
# ENGINE STATE HELPER
# ============================================================

def update_engine_state():

    global document
    global chunks

    ENGINE_STATE["ready"] = (
        "document" in globals()
        and "chunks" in globals()
        and "faiss_index" in globals()
        and "bm25_index" in globals()
    )

    if ENGINE_STATE["ready"]:

        ENGINE_STATE["document_name"] = (
            Path(document.source).name
            if getattr(document, "source", None)
            else None
        )

        ENGINE_STATE["document_source"] = (
            str(document.source)
            if getattr(document, "source", None)
            else None
        )

        ENGINE_STATE["chunk_count"] = len(chunks)


# ============================================================
# REBUILD RETRIEVAL INDEXES FOR A NEW PDF
# ============================================================

def process_uploaded_pdf(pdf_path: str):

    global document
    global chunks
    global chunk_embeddings
    global faiss_index
    global bm25_index

    print()
    print("=" * 80)
    print("PROCESSING UPLOADED RESEARCH PAPER")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Extract PDF
    # --------------------------------------------------------

    print("1. Extracting PDF...")

    document = extract_pdf(
        pdf_path
    )

    print(
        "   Pages:",
        len(document.pages)
    )

    # --------------------------------------------------------
    # 2. Chunk PDF
    # --------------------------------------------------------

    print("2. Creating chunks...")

    chunks = create_chunks(
        document,
        chunk_size=400,
        overlap=80
    )

    print(
        "   Chunks:",
        len(chunks)
    )

    if not chunks:

        raise ValueError(
            "No text could be extracted from the uploaded PDF."
        )

    # --------------------------------------------------------
    # 3. Generate embeddings
    # --------------------------------------------------------

    print("3. Generating embeddings...")

    chunk_texts = [
        chunk.text
        for chunk in chunks
    ]

    chunk_embeddings = embed_texts(
        chunk_texts
    )

    print(
        "   Embedding shape:",
        chunk_embeddings.shape
    )

    # --------------------------------------------------------
    # 4. Build FAISS index
    # --------------------------------------------------------

    print("4. Building FAISS index...")

    embedding_dimension = (
        chunk_embeddings.shape[1]
    )

    faiss_index = faiss.IndexFlatIP(
        embedding_dimension
    )

    faiss_index.add(
        np.asarray(
            chunk_embeddings,
            dtype="float32"
        )
    )

    print(
        "   Vectors indexed:",
        faiss_index.ntotal
    )

    # --------------------------------------------------------
    # 5. Build BM25 index
    # --------------------------------------------------------

    print("5. Building BM25 index...")

    bm25_corpus = [
        tokenize_for_bm25(chunk.text)
        for chunk in chunks
    ]

    bm25_index = BM25Okapi(
        bm25_corpus
    )

    print(
        "   Documents indexed:",
        len(bm25_corpus)
    )

    # --------------------------------------------------------
    # 6. Update state
    # --------------------------------------------------------

    update_engine_state()

    print()
    print("AI ENGINE READY")
    print("=" * 80)

    return {
        "document": Path(pdf_path).name,
        "pages": len(document.pages),
        "chunks": len(chunks),
        "vectors": int(faiss_index.ntotal)
    }


# ============================================================
# HEALTH ENDPOINT
# ============================================================

@app.get("/")
def root():

    update_engine_state()

    return {
        "service": "Research Retrieval & Reasoning Engine",
        "status": "running",
        "engine_ready": ENGINE_STATE["ready"]
    }


# ============================================================
# HEALTH / STATUS ENDPOINT
# ============================================================

@app.get("/health")
def health():

    update_engine_state()

    return {
        "status": "UP",
        "engine_ready": ENGINE_STATE["ready"],
        "document": ENGINE_STATE["document_name"],
        "chunks": ENGINE_STATE["chunk_count"]
    }


# ============================================================
# UPLOAD RESEARCH PAPER
# ============================================================

@app.post("/upload")
async def upload_pdf(
    file: UploadFile = File(...)
):

    if not file.filename:

        raise HTTPException(
            status_code=400,
            detail="Filename is required."
        )

    if not file.filename.lower().endswith(".pdf"):

        raise HTTPException(
            status_code=400,
            detail="Only PDF files are supported."
        )

    safe_filename = Path(
        file.filename
    ).name

    pdf_path = (
        API_UPLOAD_DIR /
        safe_filename
    )

    try:

        with open(
            pdf_path,
            "wb"
        ) as output_file:

            shutil.copyfileobj(
                file.file,
                output_file
            )

        result = process_uploaded_pdf(
            str(pdf_path)
        )

        return {
            "status": "success",
            "message": "Research paper uploaded and indexed.",
            **result
        }

    except Exception as exception:

        raise HTTPException(
            status_code=500,
            detail=f"Failed to process PDF: {str(exception)}"
        )


# ============================================================
# LOCAL RAG RETRIEVAL
# ============================================================
@app.post("/retrieve")
def retrieve_endpoint(request: RetrievalRequest):

    if not ENGINE_STATE["ready"]:
        raise HTTPException(
            status_code=400,
            detail="No research paper has been uploaded yet."
        )

    if not request.query.strip():
        raise HTTPException(
            status_code=400,
            detail="Query cannot be empty."
        )

    try:

        result = rag_query(
            query=request.query,
            candidate_k=request.candidate_k,
            top_k=request.topK
        )

        return result

    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=f"Retrieval failed: {str(e)}"
        )

# ============================================================
# EXTERNAL RESEARCH DISCOVERY
# ============================================================

@app.post("/discover")
def discover_endpoint(
    request: DiscoveryRequest
):

    update_engine_state()

    if not ENGINE_STATE["ready"]:

        raise HTTPException(
            status_code=400,
            detail=(
                "AI engine is not ready. "
                "Upload a research paper first."
            )
        )

    try:

        result = discover_similar_research(
            document=document,
            num_queries=request.num_queries,
            results_per_query=request.results_per_query,
            candidate_k=request.candidate_k,
            top_k=request.top_k,
            explain_top=request.explain_top
        )

        papers = []

        for paper in result.get(
            "papers",
            []
        ):

            papers.append({
                "title": paper.get(
                    "title"
                ),
                "authors": paper.get(
                    "authors",
                    []
                ),
                "year": paper.get(
                    "year"
                ),
                "abstract": paper.get(
                    "abstract"
                ),
                "doi": paper.get(
                    "doi"
                ),
                "url": paper.get(
                    "url"
                ),
                "source": paper.get(
                    "source"
                ),
                "embedding_similarity": paper.get(
                    "embedding_similarity"
                ),
                "rerank_score": paper.get(
                    "rerank_score"
                ),
                "relationship_analysis": paper.get(
                    "relationship_analysis"
                )
            })

        return {
            "status": "success",
            "queries": result.get(
                "queries",
                []
            ),
            "papers": papers
        }

    except Exception as exception:

        raise HTTPException(
            status_code=500,
            detail=f"Research discovery failed: {str(exception)}"
        )


# ============================================================
# STARTUP MESSAGE
# ============================================================

update_engine_state()

print()
print("=" * 80)
print("FASTAPI AI ENGINE CONFIGURED")
print("=" * 80)
print()
print("Engine ready:", ENGINE_STATE["ready"])
print("Document:", ENGINE_STATE["document_name"])
print("Chunks:", ENGINE_STATE["chunk_count"])
print()
print("Endpoints:")
print("  GET  /")
print("  GET  /health")
print("  POST /upload")
print("  POST /retrieve")
print("  POST /discover")
print()

In [108]:
# ============================================================
# START FASTAPI SERVER
# ============================================================

import uvicorn
import threading
import time

def run_fastapi():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )


fastapi_thread = threading.Thread(
    target=run_fastapi,
    daemon=True
)

fastapi_thread.start()

time.sleep(3)

print()
print("=" * 80)
print("FASTAPI SERVER STARTED")
print("=" * 80)
print()
print("Local URL:")
print("http://127.0.0.1:8000")
print()
print("Swagger:")
print("http://127.0.0.1:8000/docs")

INFO:     Started server process [1041]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



FASTAPI SERVER STARTED

Local URL:
http://127.0.0.1:8000

Swagger:
http://127.0.0.1:8000/docs


In [109]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/health"
)

print(response.status_code)
print(response.json())

INFO:     127.0.0.1:35868 - "GET /health HTTP/1.1" 200 OK
200
{'status': 'UP', 'engine_ready': True, 'document': 'ravisankarmanem_resume_wellfound.pdf', 'chunks': 4}


In [110]:
response = requests.post(
    "http://127.0.0.1:8000/retrieve",
    json={
        "query": "What model architecture was used for blockchain wallet risk prediction?",
        "candidate_k": 10,
        "top_k": 5
    }
)

print(response.status_code)
print(response.json())

INFO:     127.0.0.1:35882 - "POST /retrieve HTTP/1.1" 500 Internal Server Error
500
{'detail': "Retrieval failed: 'RetrievalRequest' object has no attribute 'topK'"}


In [ ]:
# ============================================================
# EXPOSE THE FASTAPI SERVER FOR SPRING BOOT (ngrok)
# ============================================================
# This reuses the SAME app/server started above - it does not
# redefine `app` or start a second uvicorn instance, so there is
# only ever one server bound to port 8000.

from pyngrok import ngrok
from google.colab import userdata

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")

if not NGROK_AUTH_TOKEN:
    raise ValueError("Add NGROK_AUTH_TOKEN to Colab Secrets.")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Kill any previous tunnels left over from an earlier run in this
# same Colab session, so restarting this cell doesn't pile up tunnels.
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

public_url = ngrok.connect(AI_ENGINE_PORT, "http")

print()
print("=" * 80)
print("FASTAPI PUBLIC URL (put this in Spring Boot's application.yml)")
print("=" * 80)
print()
print(f"ai-engine.base-url: {public_url.public_url}")
print()
print("This URL changes every time this cell / the Colab runtime")
print("restarts - update AI_ENGINE_BASE_URL in Spring Boot accordingly.")
